# 01 · Comparing decision criteria

Exploratory walkthrough of the budget allocation models on the synthetic sample dataset. See `docs/model.md` for the formulation.

In [ ]:
import pandas as pd
from budget_opt import load_problem, solve_scenario, solve_expected, solve_robust

problem = load_problem('../data/sample/budget_items.csv', '../data/sample/departments.csv', total_budget=1000)
problem.items.head()

## Effect of each plan under every scenario

Rows are plans (criteria), columns are the scenario that actually happens. The robust plan has the highest worst case and equalises the effect across scenarios.

In [ ]:
plans = {
    'recession': solve_scenario(problem, 'recession'),
    'stable': solve_scenario(problem, 'stable'),
    'growth': solve_scenario(problem, 'growth'),
    'expected': solve_expected(problem),
    'robust': solve_robust(problem),
}
payoff = pd.DataFrame({k: v.effects for k, v in plans.items()}).T.round(1)
payoff['worst_case'] = payoff.min(axis=1)
payoff

## Regret matrix (Savage criterion)

Regret = best achievable effect in a scenario minus the effect of the plan.

In [ ]:
regret = (payoff.drop(columns='worst_case').max() - payoff.drop(columns='worst_case')).round(1)
regret['max_regret'] = regret.max(axis=1)
regret.sort_values('max_regret')

## Where the robust plan differs from the expected-value plan

In [ ]:
diff = plans['expected'].to_frame(problem)[['item_name', 'department']].copy()
diff['expected'] = plans['expected'].allocation.values.round(1)
diff['robust'] = plans['robust'].allocation.values.round(1)
diff['delta'] = (diff['robust'] - diff['expected']).round(1)
diff[diff['delta'].abs() > 0.05].sort_values('delta')